# 03 · Motor 2 (Despachos) + Motor 3 (Producción)

### Motor 2 — ¿Qué despachar a cada tienda esta semana?
- Calcula cobertura actual vs. forecast de 4 semanas
- Genera lista de resurtido priorizando por GMROII
- Detecta excesos para redistribución

### Motor 3 — ¿Qué producir y cuánto?
- Agrega demanda de las 89 tiendas a 16 semanas
- Recomienda unidades por tipo de producto
- Distribuye por tallas según curva histórica

### Prerequisito
Haber corrido exitosamente `02_modelo_forecast.ipynb`

In [ ]:
import pandas as pd
import numpy as np
import ast, os, sys, warnings
from pathlib import Path
from datetime import date, timedelta
from sqlalchemy import text
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('.').resolve()))
from conexion import get_engine

engine = get_engine()

# ── Parámetros del negocio (ajustar según la empresa) ─────
COBERTURA_MIN_SEM  = 2.5   # semanas mínimas de stock antes de resurtir
COBERTURA_OBJ_SEM  = 6.0   # semanas objetivo al resurtir
COBERTURA_MAX_SEM  = 8.0   # semanas máximas antes de alertar exceso
GMROII_MINIMO      = 1.8   # mínimo $1.8 de margen por $1 invertido
SEMANAS_PRODUCCION = 16    # lead time total manufactura
STOCK_SEGURIDAD    = 0.15  # 15% de colchón sobre la demanda proyectada

print('✅ Configuración cargada')
print(f'   Cobertura mínima   : {COBERTURA_MIN_SEM} semanas')
print(f'   Cobertura objetivo : {COBERTURA_OBJ_SEM} semanas')
print(f'   GMROII mínimo      : {GMROII_MINIMO}')
print(f'   Lead time prod.    : {SEMANAS_PRODUCCION} semanas')

## MOTOR 2 · Recomendaciones de despacho

In [ ]:
# Último inventario disponible
df_inv = pd.read_sql("""
    SELECT i.tienda_id, i.tipo_producto, i.familia,
           i.unidades_disponibles, i.unidades_transito,
           i.cobertura_semanas, i.alerta_stock,
           i.costo_unitario, i.valor_inventario,
           t.nombre_tienda, t.ciudad, t.formato, t.indice_rotacion
    FROM fact_inventario_semanal i
    JOIN dim_tiendas t ON t.tienda_id = i.tienda_id
    WHERE i.fecha = (SELECT MAX(fecha) FROM fact_inventario_semanal)
""", engine)

# Forecast próximas 4 semanas agregado
df_fc4 = pd.read_sql("""
    SELECT tienda_id, tipo_producto, familia,
           SUM(forecast_medio) AS forecast_4sem,
           SUM(forecast_alto)  AS forecast_4sem_alto
    FROM output_forecast_semanal
    WHERE fecha_ejecucion = (SELECT MAX(fecha_ejecucion) FROM output_forecast_semanal)
      AND semana_objetivo <= (SELECT MAX(fecha_ejecucion) FROM output_forecast_semanal)
                              + INTERVAL '4 weeks'
    GROUP BY tienda_id, tipo_producto, familia
""", engine)

df_tipos = pd.read_sql('SELECT * FROM dim_tipos_producto', engine)
print(f'Inventario: {len(df_inv):,} | Forecast 4sem: {len(df_fc4):,}')

In [ ]:
# Calcular GMROII proyectado para un despacho
precio_dict = df_tipos.set_index('tipo_producto')['precio_regular'].to_dict()
costo_dict  = df_tipos.set_index('tipo_producto')['costo_produccion'].to_dict()
margen_dict = {k: (precio_dict[k]-costo_dict[k])/precio_dict[k] for k in precio_dict}

def gmroii(tipo, unidades_despachar, forecast_4sem):
    if tipo not in precio_dict or unidades_despachar <= 0: return 0
    ventas_proy = min(unidades_despachar, forecast_4sem)
    margen_proy = ventas_proy * precio_dict[tipo] * margen_dict[tipo]
    costo_inv   = unidades_despachar * costo_dict[tipo]
    return round(margen_proy / costo_inv, 2) if costo_inv > 0 else 0

# Merge y calcular necesidad
df_m = df_inv.merge(df_fc4[['tienda_id','tipo_producto','forecast_4sem','forecast_4sem_alto']],
                    on=['tienda_id','tipo_producto'], how='left').fillna({'forecast_4sem':0,'forecast_4sem_alto':0})

df_m['venta_prom_sem'] = df_m['forecast_4sem'] / 4
df_m['cobertura_real'] = ((df_m['unidades_disponibles'] + df_m['unidades_transito'])
                          / df_m['venta_prom_sem'].replace(0, np.nan)).fillna(99).round(1)

df_m['necesita_resurtido'] = df_m['cobertura_real'] < COBERTURA_MIN_SEM
df_m['tiene_exceso']       = df_m['cobertura_real'] > COBERTURA_MAX_SEM

df_m['unidades_sugeridas'] = np.where(
    df_m['necesita_resurtido'],
    (COBERTURA_OBJ_SEM * df_m['venta_prom_sem'] - df_m['unidades_disponibles']).clip(lower=0).round(0).astype(int),
    0
)
print(f'Tiendas que necesitan resurtido : {df_m.necesita_resurtido.sum():,}')
print(f'Tiendas con exceso de inventario: {df_m.tiene_exceso.sum():,}')

In [ ]:
# Construir lista de despachos con GMROII
despachos = df_m[df_m['unidades_sugeridas'] > 0].copy()
despachos['gmroii_proyectado'] = despachos.apply(
    lambda r: gmroii(r['tipo_producto'], r['unidades_sugeridas'], r['forecast_4sem']), axis=1
)
despachos['tipo_despacho']   = 'RESURTIDO'
despachos['estado']          = np.where(despachos['gmroii_proyectado'] >= GMROII_MINIMO, 'APROBADO', 'REVISAR')
despachos['fecha_ejecucion'] = date.today()
despachos['semana_despacho'] = date.today() + timedelta(days=7)
despachos['cobertura_proyectada'] = (despachos['cobertura_real'] + COBERTURA_OBJ_SEM).round(1)

print(f'\n📦 Despachos recomendados : {len(despachos):,}')
print(f'   ✅ Aprobados (GMROII ≥ {GMROII_MINIMO}): {(despachos.estado=="APROBADO").sum():,}')
print(f'   ⚠️  A revisar: {(despachos.estado=="REVISAR").sum():,}')

In [ ]:
# Guardar en Supabase
cols = ['fecha_ejecucion','tienda_id','tipo_producto','tipo_despacho',
        'unidades_sugeridas','semana_despacho','cobertura_real',
        'cobertura_proyectada','gmroii_proyectado','estado']

with engine.begin() as conn:
    conn.execute(text("DELETE FROM output_despachos_recomendados WHERE fecha_ejecucion = CURRENT_DATE"))

despachos.rename(columns={'cobertura_real':'cobertura_actual'})[cols].to_sql(
    'output_despachos_recomendados', engine, if_exists='append', index=False, method='multi', chunksize=500)
print('✅ Despachos guardados en Supabase')

# Export Excel para el equipo de planeación
os.makedirs('../outputs', exist_ok=True)
despachos[cols + ['nombre_tienda','ciudad','formato']].sort_values(
    ['estado','gmroii_proyectado'], ascending=[True,False]
).to_excel('../outputs/despachos_semana.xlsx', index=False)
print('✅ Excel: outputs/despachos_semana.xlsx')

In [ ]:
# Mapa de calor: cobertura por tipo × ciudad
heat = df_m.groupby(['tipo_producto','ciudad'])['cobertura_real'].mean().reset_index()
pivot = heat.pivot(index='tipo_producto', columns='ciudad', values='cobertura_real')
fig = px.imshow(pivot.round(1),
                title='📊 Cobertura de inventario (semanas) por tipo de producto y ciudad',
                color_continuous_scale=[[0,'#d32f2f'],[0.2,'#ff9800'],[0.45,'#4caf50'],[0.7,'#ffeb3b'],[1,'#f44336']],
                zmin=0, zmax=12, text_auto=True, aspect='auto')
fig.show()

# Distribución GMROII
fig2 = px.histogram(despachos[despachos['gmroii_proyectado']>0], x='gmroii_proyectado',
                    color='estado', nbins=30,
                    title='Distribución GMROII proyectado por despacho',
                    color_discrete_map={'APROBADO':'#4caf50','REVISAR':'#ff9800'})
fig2.add_vline(x=GMROII_MINIMO, line_dash='dash', line_color='red',
               annotation_text=f'Mínimo: {GMROII_MINIMO}')
fig2.show()

## MOTOR 3 · Recomendaciones de producción

In [ ]:
# Forecast agregado semanas 5-12 (horizonte de producción)
df_fc_prod = pd.read_sql("""
    SELECT tipo_producto, familia,
           SUM(forecast_medio) AS demanda_8sem,
           SUM(forecast_alto)  AS demanda_alta_8sem
    FROM output_forecast_semanal
    WHERE fecha_ejecucion = (SELECT MAX(fecha_ejecucion) FROM output_forecast_semanal)
    GROUP BY tipo_producto, familia
""", engine)

# Inventario actual disponible en toda la red
df_inv_red = pd.read_sql("""
    SELECT tipo_producto, SUM(unidades_disponibles) AS inv_total_red
    FROM fact_inventario_semanal
    WHERE fecha = (SELECT MAX(fecha) FROM fact_inventario_semanal)
    GROUP BY tipo_producto
""", engine)

df_prod_plan = df_fc_prod.merge(df_inv_red, on='tipo_producto', how='left').fillna({'inv_total_red':0})
df_prod_plan = df_prod_plan.merge(df_tipos[['tipo_producto','costo_produccion','tallas_json']], 
                                   on='tipo_producto', how='left')

df_prod_plan['stock_seguridad']   = (df_prod_plan['demanda_8sem'] * STOCK_SEGURIDAD).round(0)
df_prod_plan['unidades_producir'] = (
    df_prod_plan['demanda_8sem'] + df_prod_plan['stock_seguridad'] - df_prod_plan['inv_total_red']
).clip(lower=0).round(0).astype(int)

hoy = date.today()
df_prod_plan['semana_inicio_prod']  = hoy + timedelta(weeks=1)
df_prod_plan['semana_llegada_cedi'] = hoy + timedelta(weeks=SEMANAS_PRODUCCION)

print(df_prod_plan[['tipo_producto','demanda_8sem','inv_total_red',
                    'stock_seguridad','unidades_producir']].round(0).to_string(index=False))

In [ ]:
# Distribución de tallas por tipo de producto
prod_rows = []
for _, row in df_prod_plan.iterrows():
    try:
        tallas = ast.literal_eval(str(row['tallas_json']))
    except:
        tallas = {'S':0.25,'M':0.35,'L':0.25,'XL':0.15}

    total = row['unidades_producir']
    dist  = {t: int(total * p) for t, p in tallas.items()}

    prod_rows.append({
        'fecha_ejecucion':     date.today(),
        'tipo_producto':       row['tipo_producto'],
        'familia':             row['familia'],
        'semana_inicio_prod':  row['semana_inicio_prod'],
        'semana_llegada_cedi': row['semana_llegada_cedi'],
        'unidades_totales':    total,
        'distribucion_tallas': str(dist),
        'inversion_estimada':  int(total * row['costo_produccion']),
        'zona_pipeline':       'AZUL',
        'estado':              'RECOMENDADO',
    })

df_prod = pd.DataFrame(prod_rows)

with engine.begin() as conn:
    conn.execute(text("DELETE FROM output_produccion_recomendada WHERE fecha_ejecucion = CURRENT_DATE"))
df_prod.to_sql('output_produccion_recomendada', engine, if_exists='append', index=False, method='multi', chunksize=500)
print('✅ Producción guardada en Supabase')

df_prod.to_excel('../outputs/produccion_recomendada.xlsx', index=False)
print('✅ Excel: outputs/produccion_recomendada.xlsx')

In [ ]:
# Gráfica de inversión por tipo
df_graf = df_prod[df_prod['unidades_totales']>0].sort_values('inversion_estimada')
fig = px.bar(df_graf, x='inversion_estimada', y='tipo_producto', orientation='h',
             color='familia', title='💰 Inversión estimada en producción por tipo de producto',
             labels={'inversion_estimada':'Inversión (COP)','tipo_producto':''})
fig.update_layout(height=480)
fig.show()

In [ ]:
# ── RESUMEN EJECUTIVO SEMANAL ─────────────────────────────
print('='*55)
print('  RESUMEN EJECUTIVO — MODELO PREDICTIVO MODA')
print(f'  Fecha de ejecución: {date.today()}')
print('='*55)
print('\n📈 MOTOR 1 — Forecast de ventas')
print(f'   Semanas predichas    : 8')
print(f'   Tiendas cubiertas    : {df_fc4.tienda_id.nunique()}')
print(f'   Tipos de producto    : {df_fc4.tipo_producto.nunique()}')
print(f'   Unidades proyectadas : {df_fc4.forecast_4sem.sum():,.0f} (próx. 4 sem)')
print('\n📦 MOTOR 2 — Despachos recomendados')
print(f'   Total despachos      : {len(despachos):,}')
print(f'   Aprobados GMROII OK  : {(despachos.estado=="APROBADO").sum():,}')
print(f'   A revisar            : {(despachos.estado=="REVISAR").sum():,}')
print(f'   Tiendas con exceso   : {df_m.tiene_exceso.sum():,}')
print('\n🏭 MOTOR 3 — Producción recomendada')
print(f'   Tipos a producir     : {len(df_prod[df_prod.unidades_totales>0]):,}')
print(f'   Unidades totales     : {df_prod.unidades_totales.sum():,}')
print(f'   Inversión estimada   : ${df_prod.inversion_estimada.sum():,.0f} COP')
print(f'   Llegada al CEDI      : {hoy + timedelta(weeks=SEMANAS_PRODUCCION)}')
print('='*55)
print('\n✅ Archivos generados en outputs/')
print('   - despachos_semana.xlsx')
print('   - produccion_recomendada.xlsx')